Responsibilities:

ONLY:

- read raw files from volume
- 
- add metadata columns
- 
- write bronze tables.
- 
- NO transformation.

In [0]:
raw_base_path = "/Volumes/e_comm_databricks/default/olist_raw_volume/"

In [0]:
bronze_config = [

 {"table":"orders","file":"olist_orders_dataset.csv"},
 {"table":"customers","file":"olist_customers_dataset.csv"},
 {"table":"products","file":"olist_products_dataset.csv"},
 {"table":"order_items","file":"olist_order_items_dataset.csv"},
 {"table":"order_payments","file":"olist_order_payments_dataset.csv"},
 {"table":"order_reviews","file":"olist_order_reviews_dataset.csv"},
 {"table":"sellers","file":"olist_sellers_dataset.csv"},
 {"table":"geolocation","file":"olist_geolocation_dataset.csv"}

]



In [0]:
from pyspark.sql.functions import current_timestamp, col, to_date


In [0]:
def ingest_to_bronze(table_name, file_name):

    df = (
        spark.read
        .format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load(raw_base_path + file_name)
    )

    df = df \
        .withColumn("_ingest_timestamp", current_timestamp()) \
        .withColumn("_source_file", col("_metadata.file_path")) \
        .withColumn("_ingest_date", to_date(current_timestamp()))

    df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"e_comm_databricks.pipeline_bronze.{table_name}")


In [0]:
for item in bronze_config:

    print(f"Ingesting {item['table']}")

    ingest_to_bronze(item["table"], item["file"])


In [0]:
%skip
def ingest_to_bronze_incremental(table_name, file_name):

    df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", True)
        .option("inferSchema", True)
        .load(raw_base_path + file_name)
    )

    df = df \
        .withColumn("_ingest_timestamp", current_timestamp()) \
        .withColumn("_ingest_date", to_date(current_timestamp()))

    (
        df.writeStream
        .format("delta")
        .option("checkpointLocation", f"/tmp/checkpoints/{table_name}")
        .trigger(availableNow=True)
        .toTable(f"e_comm_databricks.pipeline_bronze.{table_name}")
    )
